In [2]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
# Select Device According to Availability
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print("Device selected:", device)

Device selected: cuda


In [4]:
class Window:
    def __init__(self, video_path, window_size=4, stride=2, frame_average_count=5):
        """
        A class to manage sliding windows over video frames.

        :param video_path (str): The path to the video file.
        :param window_size (int): The size of the sliding window (default is 4).
        :param stride (int): The step size for moving the window (default is 2).
        :param frame_average_count (int): The number of frames to be averaged for each group (default is 5).
        """
        # The duration of the video in seconds
        self.__video_duration = None
        # The size of the sliding window
        self.__window_size = window_size
        # The step size for moving the window
        self.__stride = stride
        # The path to the video file
        self.__video_path = video_path
        # An array containing processed frames from the video
        self.__total_frames = self.__get_frames()
        # A sliding window of frames
        self.__windows = self.__prepare_window()
        # The index of the current window in the windowed clip
        self.__window_index = 0

    def next(self):
        """
        This method strides to the next window in the windowed clip.
        :return: numpy.ndarray: The next window in the windowed clip.
        """
        if self.has_next():
            self.__window_index += 1
            return self.__windows[self.__window_index - 1]
        else:
            print("No next window")
            return None

    def has_next(self):
        """
        This method checks if there is a next window in the windowed clip.
        :return: bool: True if there is a next window, False otherwise.
        """
        return self.__window_index < len(self.__windows)

    def current_window(self):
        """
        This method returns the current window in the windowed clip.
        :return: numpy.ndarray: The current window in the windowed clip.
        """
        return self.__windows[self.__window_index]

    def previous(self):
        """
        This method strides to the previous window in the windowed clip.
        :return: numpy.ndarray: The previous window in the windowed clip.
        """
        if self.__window_index > 0:
            self.__window_index -= 1
            return self.__windows[self.__window_index]
        else:
            print("No previous window")
            return None

    def reset(self):
        """
        This method resets the window index to the beginning of the windowed clip.
        """
        self.__window_index = 0

    def __prepare_window(self):
        """
        Prepares a sliding window of frames from the average frame of the video.

        :returns:
            numpy.ndarray: A sliding window of frames.

        Notes:
            This method divides the average frame into overlapping windows of 'window_size' frames.
            The stride parameter determines the step size for moving the window.
            The number of steps is calculated based on the average frame shape and video duration.
            The resulting windows are stored in a numpy array.
        """
        window = []
        fps = self.__total_frames.shape[0] // self.__video_duration
        steps = int(round((fps * self.__window_size) // self.__stride))

        for i in range(0, len(self.__total_frames) - steps, steps):
            window.append(self.__total_frames[i: i + steps * 2])
        return np.array(window)

    def __get_frames(self):
        """
        This method reads the video file and returns the frames

        :return:
            numpy.ndarray: An array containing processed frames from the video.
        :raises:
            IOError: If the video file cannot be read or does not exist.
        """
        video = cv2.VideoCapture(self.__video_path)
        if not video.isOpened():
            raise IOError("Error reading video file")

        frame_count = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = video.get(cv2.CAP_PROP_FPS)
        self.__video_duration = frame_count // fps  # Adjust if needed for partial frames

        frames = []
        for i in range(0, frame_count, 3):
            video.set(1, i)
            ret, frame = video.read()
            if ret:
                frame = cv2.resize(frame, (224, 224))
                frame = frame.astype("float32") / 255.0
                frames.append(frame)
        video.release()
        if len(frames) < 100:
            zero_frame = np.zeros_like(frames[-1])
            frames = frames + [zero_frame] * (100 - len(frames))
            
        return np.array(frames)

    def __iter__(self):
        """
        This method makes the object iterable.
        :return: window: The object itself.
        """
        return self  # Return self to make the object iterable

    def __next__(self):
        """
        This method returns the next window in the windowed clip. (used for iteration)
        :return: numpy.ndarray: The next window in the windowed clip.
        """
        if self.has_next():
            self.__window_index += 1
            return self.__windows[self.__window_index - 1]
        else:
            raise StopIteration

    def get_current_window_stats(self):
        """
        This method returns the stats of the current window.

        :return:
            dict: A dictionary containing the statistics of the current window.
        """
        return {
            "index": self.__window_index,
            "frame_count": len(self.__windows[self.__window_index]),
            "frame_shape": self.__windows[self.__window_index].shape,
            "frame_dtype": self.__windows[self.__window_index].dtype,
        }

    def __repr__(self):
        return (f"Window (\n\tVideo Path = {self.__video_path}\n"
                f"\tTotal Window = {self.__window_size}\n"
                f"\tStride = {self.__stride}\n"
                f"\tNo. Frames Averaged = {self.__frame_average_count}\n"
                f"\tTotal Frames = {len(self.__total_frames)}\n"
                f"\tAveraged Frames = {len(self.__averaged_frames)}\n"
                f"\tShape = {self.__windows.shape}\n)")

    @property
    def shape(self):
        return self.__windows.shape


In [7]:
class WindowEmbedded:
    def __init__(self, windows: Window):
        self.__windows = windows
        self.__embedding_model = ResNetAnomaly().to(device)
        self.window_embeddings = self.__get_embeddings()

    # rest of the code remains the same
    def __get_embeddings(self):
        embeddings_list = []
        # for window in tqdm.tqdm(self.__windows, desc="Extracting Embeddings", total=self.__windows.shape[0]):
        for window in self.__windows:
            frames = torch.tensor(window).permute(0, 3, 1, 2).to(device)
            frame_embeddings_list = []
            for frame in frames:
                frame = frame.unsqueeze(0)
                frame = frame.to(device)
                frame_embeddings = self.__embedding_model(frame)
                frame_embeddings_list.append(frame_embeddings.flatten().cpu().detach().numpy())
            embeddings_list.append(frame_embeddings_list)
        self.window_embeddings = np.array(embeddings_list)
        return self.window_embeddings

In [8]:
from torchvision import models
import torch.nn as nn

class ResNetAnomaly(nn.Module):
    def __init__(self):
        
        super(ResNetAnomaly, self).__init__()

        # Load the pre-trained ResNet50 model without fine-tuning
        self.resnet = models.resnet50(pretrained=False)
        self.resnet.load_state_dict(torch.load('/kaggle/input/resnet50/resnet50.pth'))

        # Get the number of features from the last layer of the ResNet
        num_ftrs = self.resnet.fc.in_features

        # Modify the last layer to output 2048 features
        self.resnet.fc = nn.Linear(num_ftrs, 2048)

        # Define additional layers for feature learning
        self.additional_layers = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = x.to(device)
        x = self.resnet(x)
        # Pass the features through the additional layers for further learning
        x = self.additional_layers(x)

        return x

In [9]:
import numpy as np
import glob
from random import shuffle
import tqdm

train_path = '/kaggle/input/augmenteddataset/train/*/*.mp4'
train_paths = glob.glob(train_path)
train_labels = [1 if i.split('/')[5] == 'anomalous' else 0 for i in train_paths]

train_zip = list(zip(train_paths, train_labels))
shuffle(train_zip)
train_paths, train_labels = zip(*train_zip)

train_labels = list(train_labels)

In [10]:
def get_all_video_embeddings(device):
    valid_labels = []
    all_video_embedding = []

    for video_path, label in zip(tqdm.tqdm(train_paths), train_labels):
        try:
            window = Window(video_path)
            window_embed_object = WindowEmbedded(window)
            embeddings = window_embed_object.window_embeddings
            all_video_embedding.append(embeddings)
            valid_labels.append(label)
        except ValueError as e:
            print(f"Skipping video {video_path} due to error: {e}")
            continue
    
    return all_video_embedding, valid_labels

In [ ]:
# from keras.preprocessing.sequence import pad_sequences

# max_sequence_length = max(window.shape[0] for video_embedding in all_video_embedding for window in video_embedding)

# all_video_embedding_padded = []

# for video_embedding in all_video_embedding:
#     video_embedding_padded = []
#     for window in video_embedding:
#         if window.shape[0] != max_sequence_length:
#             # Apply padding only if the window shape does not match the max_sequence_length
#             window_padded = pad_sequences([window], maxlen=max_sequence_length, padding='post', dtype='float32')[0]
#             video_embedding_padded.append(window_padded)
#         else:
#             video_embedding_padded.append(window)
#     all_video_embedding_padded.append(video_embedding_padded)

# all_video_embedding_padded = np.array(all_video_embedding_padded)


In [ ]:
all_video_embedding, valid_labels = get_all_video_embeddings(device)

np.save('/kaggle/working/train_embeddings.npy', all_video_embedding_padded) 
np.save('/kaggle/working/train_labels.npy', np.array(valid_labels))

  0%|          | 0/9150 [00:00<?, ?it/s]/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
  1%|          | 77/9150 [03:27<4:59:30,  1.98s/it]

Skipping video /kaggle/input/augmenteddataset/train/nonanomalous/Normal_Videos650_x264_003.mp4 due to error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (5,) + inhomogeneous part.


  2%|▏         | 196/9150 [08:43<4:54:11,  1.97s/it]

Skipping video /kaggle/input/augmenteddataset/train/anomalous/553_Arson044_x264_003.mp4 due to error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.


  3%|▎         | 307/9150 [13:35<4:47:44,  1.95s/it]

...

 52%|█████▏    | 4719/9150 [3:26:30<2:29:54,  2.03s/it]

Skipping video /kaggle/input/augmenteddataset/train/nonanomalous/Normal_Videos250_x264_011.mp4 due to error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.


 52%|█████▏    | 4768/9150 [3:28:38<2:24:17,  1.98s/it]

Skipping video /kaggle/input/augmenteddataset/train/anomalous/Burglary004_x264_005.mp4 due to error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (4,) + inhomogeneous part.


 52%|█████▏    | 4783/9150 [3:29:18<3:14:22,  2.67s/it]

In [10]:
!pip install sru
!pip install ninja

import torch
from sru import SRU, SRUCell
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

In [11]:
#load train labels
labels = np.load('/kaggle/working/train_labels.npy', allow_pickle=True)
print(labels.shape)
label_list = list(labels)
all_train_labels = []
for label in label_list:
    for i in range(4):
        all_train_labels.append(label)
        
np.save('/kaggle/working/all_train_labels.npy', np.array(all_train_labels))

(9047,)


In [12]:
# Select Device According to Availability
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print("Device selected:", device)

Device selected: cuda


In [5]:
# video_embeddings = np.load('/kaggle/working/train_embeddings.npy', allow_pickle=True)
# video_embeddings = video_embeddings.reshape(video_embeddings.shape[0]*video_embeddings.shape[1], video_embeddings.shape[2]*video_embeddings.shape[3])
# print(video_embeddings.shape)
# labelss = np.load('/kaggle/working/all_train_labels.npy', allow_pickle=True)
# print(labelss.shape)

# # Split your data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(video_embeddings, labelss, test_size=0.2, random_state=42)

# #convert to tensor
# video_embeddings_train = torch.from_numpy(X_train).to(device)
# labels_train = torch.from_numpy(y_train).to(device)
# print(video_embeddings_train.shape)
# print(labels_train.shape)

# video_embeddings_test = torch.from_numpy(X_test).to(device)
# labels_test = torch.from_numpy(y_test).to(device)
# print(video_embeddings_test.shape)
# print(labels_test.shape)



ValueError: cannot reshape array of size 837147292 into shape (9047,4,40,1024)

In [16]:

import numpy as np
from sklearn.model_selection import train_test_split

video_embeddings = np.load('/kaggle/working/train_embeddings.npy', allow_pickle=True)
print(video_embeddings.shape)

labelss = np.load('/kaggle/working/train_labels.npy', allow_pickle=True)
print(labelss.shape)


# # Split your data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(video_embeddings, labelss, test_size=0.2, random_state=42)

X_train_reshaped = X_train.reshape(X_train.shape[0]*X_train.shape[1], X_train.shape[2],X_train.shape[3])
X_test_reshaped = X_test.reshape(X_test.shape[0]*X_test.shape[1], X_test.shape[2],X_test.shape[3])

new_y_train = np.repeat(y_train,4)
new_y_test = np.repeat(y_test, 4)
# convert to tensor
video_embeddings_train = torch.from_numpy(X_train_reshaped).to(device)
labels_train = torch.from_numpy(new_y_train).to(device)
print(video_embeddings_train.shape)
print(labels_train.shape)

video_embeddings_test = torch.from_numpy(X_test_reshaped).to(device)
labels_test = torch.from_numpy(new_y_test).to(device)
print(video_embeddings_test.shape)
print(labels_test.shape)

ValueError: cannot reshape array of size 837147292 into shape (9047,4,40,1024)

In [6]:
# Create DataLoaders for your training and testing data
train_data = TensorDataset(video_embeddings_train, labels_train)
test_data = TensorDataset(video_embeddings_test, labels_test)

batch_size = 8
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_data, shuffle=True, batch_size=batch_size)

In [7]:
input_size, hidden_size = 40960, 1024
num_classes = 2  # fight and non-fight
num_layers = 2
dropout_prob = 0.2
l2_reg_lambda = 1e-4  # Adjust the value as needed (try different powers of 10)

class SRUModel(torch.nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, dropout_prob, l2_reg_lambda):
        super(SRUModel, self).__init__()
        self.sru = SRU(input_size, hidden_size, num_layers=num_layers, dropout=dropout_prob, bidirectional=False,
                        layer_norm=True, highway_bias=0, rescale=True)
        self.dropout = torch.nn.Dropout(dropout_prob)  # Apply dropout before the final linear layer
        self.fc = torch.nn.Linear(hidden_size, num_classes)
        self.l2_reg_lambda = l2_reg_lambda

    def forward(self, x):
        output_states, _ = self.sru(x)
        output = self.fc(self.dropout(output_states[-1]))
        return output

    def l2_regularization(self):
        l2_reg = torch.tensor(0., device=self.fc.weight.device)
        for param in self.parameters():
            l2_reg += torch.norm(param, p=2)
        return self.l2_reg_lambda * l2_reg
model = SRUModel(input_size, hidden_size, num_layers, num_classes, dropout_prob, l2_reg_lambda)
model.to(device)

# Define your loss function and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

In [8]:
num_epochs = 10
for epoch in range(num_epochs):
    total_correct = 0
    total_samples = 0
    total_loss = 0.0
    for i, (videos, labels) in enumerate(train_loader):
#         videos = videos.unsqueeze(0)
        # Forward pass
        outputs = model(videos)
        labels = labels.long()  # Convert labels to Long type
        loss = criterion(outputs, labels) + model.l2_regularization()  # calculates loss
        total_loss += loss.item()
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # Calculate accuracy per batch
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        total_correct += (predicted == labels).sum().item()
        batch_accuracy = 100 * total_correct / total_samples
#         print(f'Epoch [{epoch + 1}/{num_epochs}], Step [{i + 1}/{len(train_loader)}], Loss: {loss.item()}, Accuracy: {batch_accuracy}%')
    
    # Print accuracy per epoch
    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = 100 * total_correct / total_samples
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss}, Accuracy: {epoch_accuracy}%')

model.eval()

Epoch [1/10], Loss: 0.9117827682464319, Accuracy: 69.22279792746114%
Epoch [2/10], Loss: 0.4736756605769627, Accuracy: 84.6839378238342%
Epoch [3/10], Loss: 0.347931063314784, Accuracy: 91.78238341968913%
Epoch [4/10], Loss: 0.3210043697874004, Accuracy: 94.47322970639033%
Epoch [5/10], Loss: 0.3348600896650271, Accuracy: 95.49568221070811%
Epoch [6/10], Loss: 0.2936975621294336, Accuracy: 96.61830742659758%
Epoch [7/10], Loss: 0.36703855561004484, Accuracy: 96.15889464594127%
Epoch [8/10], Loss: 0.4520901458538116, Accuracy: 96.38341968911917%
Epoch [9/10], Loss: 0.3601827441896645, Accuracy: 97.24352331606218%
Epoch [10/10], Loss: 0.4383208026054914, Accuracy: 96.70811744386874%


SRUModel(
  (sru): SRU(
    (rnn_lst): ModuleList(
      (0): SRUCell(40960, 1024, dropout=0.2, rescale=True, layer_norm=True)
      (1): SRUCell(1024, 1024, rescale=True, layer_norm=True)
    )
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=1024, out_features=2, bias=True)
)

In [9]:
with torch.no_grad():
    correct = 0
    total = 0
    for videos, labels in test_loader:
        videos = videos.unsqueeze(0)
        outputs = model(videos)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        #print test accuracy per test

    print(f'Test Accuracy of the model on the test videos: {100 * correct / total}%')

# Save your model
torch.save(model.state_dict(), 'SRUpp_model.pth')

Test Accuracy of the model on the test videos: 98.27300359215253%
